## Diabetes Prediction
<!-- Author : Ashutosh Sahu(ashusahu198@gmail.com) -->

Dataset Source - [kaggle](https://www.kaggle.com/tigganeha4/diabetes-dataset-2019)

other references -
- https://www.sciencedirect.com/science/article/pii/S1877050920308024?via%3Dihub



## Data Dictionary
|S.no.| Parameters | Details |
|:-|:-|:-|
|1| Age | Age of the Patient (18 or above)|
|2| Gender | Male or Female |
|3| Family_Diabetes | Family history with diabetes (yes or no) |
|4| highBP | Diagnosed with high blood pressure (yes or no) |
|5| PhysicallyActive | walk/run or can be physically active |
|6| BMI | Body Mass Index |
|7| Smoking | Whether the person smokes or not (yes or no) |
|8| Alcohol | Alcohol consumer(yes or no)|
|9| Sleep | Hours of sleep |
|10| SoundSleep | Hours of sound sleep |
|11| RegularMedicine | Regular intake of medicine (yes or no) |
|12| JunkFood | Junk food consumer(yes or no)|
|13| Stress | how much stress taken |
|14| BPLevel | Hign/normal/low |
|15| Pregnancies | no. of Pregnancies |
|16| Pdiabetes | Gestation diabetes(yes or no) |
|17| UrinationFreq | Frequency of Urination (not much or quite much)|
|18| Diabetic | yes or no |

In [ ]:
import pandas
df = pandas.read_csv('/kaggle/input/diabetes-dataset-2019/diabetes_dataset__2019.csv')
df

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# !pip install --upgrade pandas_profiling 

### Handling Missing values

In [ ]:
# import sys

# !"{sys.executable}" -m pip install -U pandas-profiling[notebook]
# !jupyter nbextension enable --py widgetsnbextension

In [ ]:

# from pandas_profiling import ProfileReport
# #df.profile_report()
# pandas_profiling.ProfileReport(df)

In [ ]:
df.isna().sum()

In [ ]:
# for removing rows.
indexes = df[df['Diabetic'].isna() | df['Pdiabetes'].isna() | df['BMI'].isna()].index.to_list()

print(indexes)
df.drop(index= indexes,inplace = True)

In [ ]:
# for imputing pregnancies
print(df['Pregancies'].value_counts())
df['Pregancies'].fillna(value = 0.0, inplace= True)
print(df['Pregancies'].value_counts())
df['Pregancies'] = df['Pregancies'].astype(int)
df['Pregancies']

In [ ]:

import seaborn
print(df['Diabetic'].value_counts())
df['Diabetic'].replace(' no', 'no', inplace=True)
print(df['Diabetic'].value_counts())
seaborn.countplot(x = 'Diabetic',data = df)

In [ ]:
print(df['RegularMedicine'].value_counts())
df['RegularMedicine'].replace('o', 'no', inplace =True)
print(df['RegularMedicine'].value_counts())

In [ ]:
print(df['Pdiabetes'].value_counts())
df['Pdiabetes'].replace('0', 'no', inplace = True)
print(df['Pdiabetes'].value_counts())


### Numerical Variable Analysis

In [ ]:
import seaborn
seaborn.pairplot(data=df, hue='Diabetic')

In [ ]:
import matplotlib.pyplot as pyplot
import seaborn
fig = pyplot.figure(figsize= (12,5))
pyplot.subplot(131)
seaborn.violinplot(data = df, x = 'Diabetic', y = 'BMI')
pyplot.subplot(132)
seaborn.violinplot(data = df, x = 'Diabetic', y = 'Sleep')
pyplot.subplot(133)
seaborn.violinplot(data = df, x = 'Diabetic', y = 'SoundSleep')

### Categorical Variable Analysis

In [ ]:

cols = ['Age', 'Gender', 'Family_Diabetes', 'highBP', 'PhysicallyActive',
       'Smoking', 'Alcohol', 'RegularMedicine',
       'JunkFood', 'Stress', 'BPLevel', 'Pregancies', 'Pdiabetes',
       'UriationFreq']

pyplot.figure(figsize = (15,25))

i  = 0
for j in range(9):
    pyplot.xticks(rotation=75)
    pyplot.subplot(int(str(3)+str(3)+str(j+1)))
    seaborn.countplot(x = cols[i], hue='Diabetic',data = df)
    i += 1
pyplot.show()

pyplot.figure(figsize = (15,25))
for j in range(5):
    pyplot.xticks(rotation=75)
    pyplot.subplot(int(str(3)+str(3)+str(j+1)))
    seaborn.countplot(x = cols[i], hue='Diabetic',data = df)
    i += 1


### Encoding categorical data


In [ ]:
df = pandas.get_dummies(df, drop_first= True)

preg = pandas.get_dummies(df['Pregancies'],prefix='Pregnancies',drop_first= True)

print(preg.head())

df = pandas.concat([preg,df], axis = 1)
df.drop(columns=['Pregancies'],inplace=True)

In [ ]:
df

In [ ]:
df.columns

### Scaling

In [ ]:
df[['BMI', 'Sleep','SoundSleep']].hist(figsize=(20,10))
pyplot.show()

Standardizing all features since they show a gaussian distribution.

In [ ]:
standard_df = df.copy()
# y = standard_df['Diabetic_yes']
# x = standard_df.drop(columns = ['Diabetic_yes'])

from sklearn.preprocessing import StandardScaler
cols = ['BMI','Sleep','SoundSleep']
standard_df[cols] = StandardScaler().fit_transform(standard_df[cols].values)
standard_df


## Train Test Split

In [ ]:
y = df['Diabetic_yes']
x = df.drop(columns= ['Diabetic_yes'])

stan_y = standard_df['Diabetic_yes']
stan_x = standard_df.drop(columns=['Diabetic_yes'])

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)

stan_x_train, stan_x_test, stan_y_train, stan_y_test = train_test_split(stan_x, stan_y, test_size = 0.25, random_state = 42)

### OverSampling
__ADASYN__ : ADASYN(Adaptive Synthetic) is a generalized form of the SMOTE algorithm. This algorithm aims to oversample the minority class by generating synthetic instances for it. It considers the density distribution, which decides the no. of synthetic instances generated for samples which difficult to learn. Due to this, it helps in adaptively changing the decision boundaries based on the samples difficult to learn. This is better than SMOTE.

In [ ]:
# oversampling using adasyn
cols = x_train.columns

from imblearn.over_sampling import ADASYN

ada = ADASYN(random_state = 42)
stan_x_train, stan_y_train = ada.fit_resample(stan_x_train,stan_y_train)



In [ ]:
import numpy
print(numpy.unique(y_train,return_counts= True))
print(numpy.unique(stan_y_train,return_counts=True))

## Training Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import classification_report

In [ ]:
models = [
    LogisticRegression(random_state = 42, max_iter = 1000),
    SGDClassifier(random_state = 42),
    GaussianNB(),
    KNeighborsClassifier(),
    LinearDiscriminantAnalysis(),
    SVC(random_state = 42, probability = True,verbose = 2),
    DecisionTreeClassifier(random_state = 42),
    RandomForestClassifier(random_state = 42),
    GradientBoostingClassifier(random_state = 42),
    LGBMClassifier(random_state = 42)
]

parameters = [
    ['logreg_params' , {'penalty':['l1', 'l2', 'elasticnet', 'none'],
                        'C': numpy.logspace(0,4,32),
                        'solver' : ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
                        'max_iter' : numpy.linspace(50,300),
                        'multi_class': ['auto', 'ovr', 'multinomial'],
                        'l1_ratio' : numpy.linspace(0,1,8),
                        }],
    ['sgdc_params' , {'loss' : ['hinge', 'log', 'modified_huber', 'squared_hinge', 'perceptron','squared_loss', 'huber', 'epsilon_insensitive','squared_epsilon_insensitive'],
                      'penalty' : ['l2', 'l1', 'elasticnet'],
                      'alpha' : numpy.linspace(0.0001,0.1,100),
                      'l1_ratio' : numpy.linspace(0,1,50),
                      'learning_rate' : ['constant', 'optimal','invscaling','adaptive'],
                      'eta0' : numpy.linspace(0.0001, 0.05, 50),
                      'power_t' : numpy.linspace(0,1, 10),
                      'average' :[True, False]
                      }],
    ['gnb_params' , {}],
    ['knn_params' , {'n_neighbors': list(range(1,50)),
                     'weights':['uniform', 'distance'],
                     'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                     'leaf_size' : list(range(1,100)),
                     'p' :[1,2,3],
                     'metric' : ['euclidean','manhattan','chebyshev','minkowski','seuclidean','mahalanobis']
                     }],
    ['lda_params' , {'solver' : ['svd', 'lsqr', 'eigen'],
                     'shrinkage' : ['auto',0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
                     'store_covariance' :[True,False]

                     }],
    ['svc_params' , {'C': range(25,35,2),
                     'degree' : list(range(1,6)),
                     'kernel' : ['linear', 'poly', 'rbf', 'sigmoid'],
                     'gamma':['scale', 'auto', 0.001, 0.01, 0.1, 1],
                     'shrinking' :[True,False],
                     'decision_function_shape': ['ovo', 'ovr'],
                     }],
    ['dt_params' , {'criterion': ['gini', 'entropy'],
                    'splitter' :['best','random'],
                    'max_depth' : range(1,20),
                    'min_samples_split' : range(2,500,10),
                    'min_samples_leaf' : range(1,6),
                    'max_features' : ['auto', 'log2', None]
                    }],
    ['rf_params' , {'n_estimators' : list(range(10,300,10)),
                    'criterion' : ['gini', 'entropy'],
                    'min_samples_split' : list(range(2,20)),
                    'min_samples_leaf' : list(range(1,20)),
                    'max_features' :['auto','log2',None],
                    'max_leaf_nodes' :list(range(1,20)),
                    'bootstrap' :[True,False],
                    'oob_score' : [True,False]
                    }],
    ['gbc_params' , {'loss' :['deviance', 'exponential'],
                     'learning_rate' : [0.05,0.1,0.2,0.3,0.4,0.5],
                     'n_estimators' :[100,150,200],
                     'min_samples_split' : [2,3,5,7,8,10],
                     'criterion' : ['friedman_mse', 'mse'],
                     'subsample' : [0.2,0.4,0.6,0.8,1.0],
                     'min_samples_leaf' : [1,2,3,4,5,6,7,8,9,10],
                     'max_features' :['auto','log2',None]
                    }],
    ['lgbmc_params' , {'boosting_type' : ['gbdt','dart', 'goss','rf'],
                       'num_leaves' : range(2,130,5),
                       'learning_rate' : [0.1,0.2,0.3,0.4,0.5],
                       'n_estimators' :[100,150,200],
                       'reg_alpha' : [0.1,0.2,0.3,0.4,0.5],
                       'reg_lambda' : [0.1,0.2,0.3,0.4,0.5]
                       }]
]

#### Over Ordinary data

In [ ]:

accuracy = []
          
for i in range(len(models)):
    clf = RandomizedSearchCV(models[i],
                       param_distributions= parameters[i][1],
                       cv = StratifiedKFold(n_splits = 10),
                       scoring = "accuracy",
                       n_jobs = -1, verbose = 2)
    
    clf.fit(x_train,y_train)
    y_pred = clf.predict(x_test)
    y_train_pred = clf.predict(x_train)
    accuracy.append(clf.best_score_ * 100)
    print()
    print('best_estimator :', clf.best_estimator_)
    
    print('train report : \n',classification_report(y_train, y_train_pred))
    print('test_report : \n',classification_report(y_test,y_pred))
    print("---------------------------------------------------------------------------")

In [ ]:
model_names = ['log_reg', 'sgdc','gnb','knn','lda','svc','dt','rf','gbc','lgbmc']
a = pandas.DataFrame(accuracy, columns = ['accuracy'], index = model_names)
print(a)
seaborn.barplot(data = a, y = 'accuracy', x = a.index)

#### Over standardized Data

In [ ]:
accuracy = []
    
for i in range(len(models)):
    
    clf = RandomizedSearchCV(models[i],
                       param_distributions= parameters[i][1],
                       cv = StratifiedKFold(n_splits = 10),
                       scoring = "accuracy",
                       n_jobs = -1, verbose = 2)
    
    clf.fit(stan_x_train, stan_y_train)
    y_pred = clf.predict(stan_x_test)
    y_train_pred = clf.predict(stan_x_train)
    accuracy.append(clf.best_score_ * 100)
    
    print()
    print('best_estimator :', clf.best_estimator_)
    
    print('train report : \n',classification_report(stan_y_train, y_train_pred))
    print('test_report : \n',classification_report(stan_y_test,y_pred))
    print("---------------------------------------------------------------------------")


In [ ]:
model_names = ['log_reg', 'sgdc','gnb','knn','lda','svc','dt','rf','gbc','lgbmc']
a = pandas.DataFrame(accuracy, columns = ['accuracy'], index = model_names)
print(a)
seaborn.barplot(data = a, y = 'accuracy', x = a.index)

#### Fine Tuning Best Estimator - LGBMC on unscaled data.


In [ ]:
def report(results, n_top=3):
    for i in range(1, n_top + 1):
        candidates = numpy.flatnonzero(results['rank_test_score'] == i)
        for candidate in candidates:
            print("Model with rank: {0}".format(i))
            print("Mean validation score: {0:.3f} (std: {1:.3f})"
                  .format(results['mean_test_score'][candidate],
                          results['std_test_score'][candidate]))
            print("Parameters: {0}".format(results['params'][candidate]))
            print("")

parameters = {'boosting_type' : ['gbdt' ,'goss'],
                       'num_leaves' : range(40,55,5),
                       'learning_rate' : [0.1,0.2,0.4],
                       'n_estimators' :[100,120,140,160,180,200],
                       'reg_alpha' : [0.2,0.4],
                       'reg_lambda' : [0.2,0.5]
                       }
          

clf = RandomizedSearchCV(LGBMClassifier(random_state= 42),
                    param_distributions= parameters,
                    cv = StratifiedKFold(n_splits = 5),
                    scoring = "accuracy",
                    n_jobs = -1, verbose = 2)

clf.fit(x_train,y_train)
report(clf.cv_results_)

### Final LGBMC model

In [ ]:
params = {'reg_lambda': 0.2, 'reg_alpha': 0.4, 'num_leaves': 50, 'n_estimators': 200, 'learning_rate': 0.1, 'boosting_type': 'gbdt'}
clf = LGBMClassifier(**params)
clf.fit(x_train, y_train)
y_pred = clf.predict(x_train)
print('train report\n',classification_report(y_train,y_pred))
y_pred = clf.predict(x_test)
print('test report\n',classification_report(y_test,y_pred))

#### Confusion Matrix - LGBMC model

In [ ]:
conf_matrix  = pandas.crosstab(y_test, y_pred)
seaborn.heatmap(conf_matrix,annot=True, fmt='.0f')

In [ ]:
#self inserted
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))

## Neural Network Hyperparameter random search

In [ ]:
# gridsearch cross validation in neural network model

from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.wrappers.scikit_learn import KerasClassifier

def report(results, n_top=3):
    for i in range(1, n_top + 1):
        candidates = numpy.flatnonzero(results['rank_test_score'] == i)
        for candidate in candidates:
            print("Model with rank: {0}".format(i))
            print("Mean validation score: {0:.3f} (std: {1:.3f})"
                  .format(results['mean_test_score'][candidate],
                          results['std_test_score'][candidate]))
            print("Parameters: {0}".format(results['params'][candidate]))
            print("")

def nn_model(activation = 'relu', neurons = 32, optimizer = 'Adam',dropout = 0.1, init_mode = 'uniform'):
    model = Sequential()
    model.add(Dense(32, input_dim = 32, kernel_initializer = init_mode, activation= activation))
    model.add(Dense((neurons*2)//3, kernel_initializer = init_mode,activation= activation))
    model.add(Dense((neurons*4)//9,kernel_initializer = init_mode,  activation = activation))
    model.add(Dropout(dropout))
    model.add(Dense(1, kernel_initializer = init_mode, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer= optimizer, metrics=['accuracy'])
    return model

# Defining grid parameters
activation = ['softmax', 'softplus', 'softsign', 'relu', 'selu', 'elu', 'tanh','sigmoid', 'linear']
neurons = range(31,39)
dropout = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
init_mode = ['uniform', 'lecun_uniform', 'normal', 'zero', 'glorot_normal', 'glorot_uniform', 'he_normal', 'he_uniform']
optimizer = ['SGD', 'Adam', 'Adamax','RMSprop','Adagrad','Adadelta','Nadam','Ftrl']
batch_size = range(10,101,10)
param_grid = dict(activation = activation, neurons = neurons, optimizer = optimizer, dropout = dropout, init_mode = init_mode, batch_size = batch_size)

clf = KerasClassifier(build_fn= nn_model, epochs= 10, verbose= 1)

model = RandomizedSearchCV(estimator= clf, param_distributions = param_grid, n_jobs=-1,verbose = 3)
model.fit(stan_x_train,stan_y_train)

report(model.cv_results_)

In [ ]:
activation = [ 'relu', 'selu', 'tanh']
neurons = range(37,38,39)
dropout = [0.0, 0.15, 0.1]
init_mode = ['uniform', 'normal', 'glorot_uniform']
optimizer = [ 'Adamax','RMSprop','Nadam']
batch_size = range(30,101,10)
param_grid = dict(activation = activation, neurons = neurons, optimizer = optimizer, dropout = dropout, init_mode = init_mode, batch_size = batch_size)

clf = KerasClassifier(build_fn= nn_model, epochs= 10, verbose= 1)

model = RandomizedSearchCV(estimator= clf, param_distributions = param_grid, n_jobs=-1,verbose = 3)
model.fit(stan_x_train,stan_y_train)

report(model.cv_results_)

### Final Neural Network

In [ ]:
model = nn_model(optimizer='Nadam', neurons = 37, init_mode='glorot_uniform', dropout=0.15,activation='relu') 
model.fit(stan_x_train, stan_y_train, batch_size = 40, epochs = 90)

In [ ]:
y_pred = model.predict_classes(stan_x_train)
print('train report\n', classification_report(stan_y_train, y_pred))

y_pred = model.predict_classes(stan_x_test)
print('test report\n', classification_report(stan_y_test, y_pred))

In [ ]:
#self inserted
from sklearn.metrics import accuracy_score
accuracy_score(stan_y_test, y_pred)

#### Confusion Matrix - Neural Network

In [ ]:
conf_matrix  = pandas.crosstab(stan_y_test, y_pred.ravel())
seaborn.heatmap(conf_matrix,annot=True, fmt='.0f')


## Conclusions:
LGBMC Model perfoms better than Neural Network

In [ ]:
import pickle

params = {'reg_lambda': 0.2, 'reg_alpha': 0.4, 'num_leaves': 50, 'n_estimators': 200, 'learning_rate': 0.1, 'boosting_type': 'gbdt'}
model = LGBMClassifier(**params)
model.fit(x, y)

In [ ]:

# save the model to disk
filename = 'diabetes_model.pickle'
pickle.dump(model, open(filename, 'wb'))
 
# some time later...
 
# load the model from disk
# loaded_model = pickle.load(open(filename, 'rb'))
# result = loaded_model.score(X_test, Y_test)
# print(result)

